In [1]:
import pandas as pd
import os

In [2]:
def analyze_model_results(file_paths):
    all_results = []

    # Step 1: Read each file and add a column to identify the model type
    for model_name, path in file_paths.items():
        try:
            # Read the CSV file into a DataFrame
            df = pd.read_csv(path)
            # Add a new column to specify the source model/script
            df['Model_Type'] = model_name
            all_results.append(df)
            print(f"Successfully loaded and processed '{path}'")
        except FileNotFoundError:
            print(f"Warning: File not found at '{path}'. Skipping.")
        except Exception as e:
            print(f"Warning: Could not process file '{path}'. Error: {e}. Skipping.")

    if not all_results:
        print("Error: No dataframes were loaded. Please check file paths.")
        return pd.DataFrame()
    combined_df = pd.concat(all_results, ignore_index=True)
    best_indices = combined_df.groupby('Crop')['Accuracy %'].idxmax()
    best_summary = combined_df.loc[best_indices]
    final_columns = [
        'Crop',
        'Model_Type',
        'Best n_steps',
        'Accuracy %',
        'Loss % (MAPE)',
        'Best RMSE'
    ]
    for col in final_columns:
        if col not in best_summary.columns:
            best_summary[col] = pd.NA
            
    final_summary = best_summary[final_columns]
    
    # Sort the final results by accuracy in descending order
    final_summary_sorted = final_summary.sort_values(by='Accuracy %', ascending=False).reset_index(drop=True)

    return final_summary_sorted


In [3]:
files = {
    "GRU_General": "GRU_Results/GRU_Accuracy_Summary.csv",
    "GRU_Optimized": "GRU_Results_Optimized/GRU_Accuracy_Summary_Optimized.csv",
    "LSTM_General": "LSTM_Results/LSTM_Results.csv",
    "LSTM_Optimized": "LSTM_Results_Optimized/LSTM_Results_Optimized.csv"
}
final_champion_models = analyze_model_results(files)
if not final_champion_models.empty:
    print("\n" + "="*80)
    print("                Champion Model Summary for Each Crop")
    print(" (The best performing model out of all 4 experiments for each crop)")
    print("="*80)
    print(final_champion_models.to_string())

Successfully loaded and processed 'GRU_Results/GRU_Accuracy_Summary.csv'
Successfully loaded and processed 'GRU_Results_Optimized/GRU_Accuracy_Summary_Optimized.csv'
Successfully loaded and processed 'LSTM_Results/LSTM_Results.csv'
Successfully loaded and processed 'LSTM_Results_Optimized/LSTM_Results_Optimized.csv'

                Champion Model Summary for Each Crop
 (The best performing model out of all 4 experiments for each crop)
                   Crop      Model_Type  Best n_steps  Accuracy %  Loss % (MAPE)  Best RMSE
0                Garlic     GRU_General             7       97.48           2.52     857.80
1            Cashewnuts     GRU_General            60       93.47           6.53    2871.60
2                Rubber  LSTM_Optimized            60       93.44           6.56    1034.56
3       Maize-2019-2022     GRU_General            30       92.69           7.31     204.74
4          Red_Chillies     GRU_General            14       91.43           8.57    2380.68
5   Blac